# 🎙️ Clonador de voz · XTTS-v2 en Google Colab

Convierte cualquier texto en audio **con tu propia voz**, a partir de una grabación tuya guardada en **Google Drive**. Todo se procesa en la **GPU** de Colab.

### Cómo usarlo
1. Comprueba que el entorno tenga GPU: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. El cuaderno ya viene configurado así.
2. Completa los campos de las celdas **4** (tu audio en Drive) y **5** (el texto). Luego ejecuta las celdas **1 → 5** en orden, o todas a la vez con **Ctrl + F9**. La primera vez, Colab te pedirá permiso para acceder a tu Drive.
3. Para generar otro audio con la misma voz, cambia el texto y ejecuta **solo la celda 5**. Para cambiar de voz, ejecuta la **4** y después la **5**. Si pides varias **tomas**, escúchalas y guarda la mejor con la celda **6**.
4. Colab guarda los valores de los formularios dentro del propio cuaderno: si lo abres siempre desde la misma copia de tu Drive, tus campos y ajustes se conservan de una sesión a otra.

Cada celda muestra solo sus parámetros. Para ver o editar su código, haz **doble clic sobre el formulario** de la celda (o elige *Mostrar código* en su menú **⋮**).

### Para un buen parecido
- Graba **entre 30 y 60 segundos** hablando con naturalidad, sin eco, sin música y con una sola voz.
- Sirven WAV, MP3, M4A, OGG/OPUS (notas de voz de WhatsApp) e incluso videos (MP4, MOV…). También puedes indicar una carpeta con varios audios tuyos.
- El modelo imita el tono y el ritmo de la grabación: si quieres voz de narrador, graba como narrador.
- En la celda 5 eliges el idioma principal, y puedes leer partes en otro con `[en]…[/en]` (o `[pt]`, `[fr]`, `[it]`, `[de]`…): tu voz habla en todos.
- Si el modelo lee mal una sigla, una abreviatura o un nombre, corrígelo en **Pronunciación** (celda 5), p. ej. `DNI = de ene i`.

> **Licencia y uso responsable.** XTTS-v2 se distribuye con la [Coqui Public Model License](https://tts-hub.github.io/cpml), que solo permite **uso no comercial** del modelo y de los audios que genera. El cuaderno **asume ese uso por defecto** y acepta la licencia en tu nombre (casilla `uso_no_comercial` de la celda 3, ya marcada); desmárcala si vas a darle a los audios un uso comercial. Clona únicamente tu voz o la de alguien que te haya dado su permiso.

In [ ]:
# @title 1 · Preparar el entorno (GPU y librerías) { display-mode: "form" }
# @markdown Comprueba la GPU e instala **coqui-tts**, el motor de XTTS-v2. La primera vez tarda 1–3 minutos; si la vuelves a ejecutar, no reinstala nada.

"""Celda 1 · Entorno: verifica la GPU, instala lo que falte y reinicia la sesión solo si es imprescindible."""

import importlib
import importlib.metadata as metadata
import os
import re
import shutil
import signal
import subprocess
import sys
import time

VERSION_COQUI_TTS = "0.28.0"  # versión con la que se verificó este cuaderno
PAQUETES_EXTRA: list[str] = []  # p. ej. ["transformers<5.19"] si una actualización de Colab rompe la instalación
MODULOS_SENSIBLES = ("numpy", "scipy", "torch", "torchaudio", "transformers", "tokenizers", "huggingface_hub")


class ErrorUsuario(Exception):
    """Problema que el usuario puede corregir: se muestra como un mensaje, sin traceback."""


def registrar_errores_amigables() -> None:
    """Muestra cada ErrorUsuario en una sola línea roja en lugar de un traceback.

    IPython ignora lo que devuelve el manejador al ejecutar una celda, por eso imprime el mensaje;
    la celda igual queda marcada con error y «Ejecutar todas» se detiene ahí.
    """
    try:
        shell = get_ipython()  # noqa: F821 (la define IPython/Colab)
    except NameError:
        return
    if shell is None:
        return

    def mostrar(_shell, _tipo, valor, _traza, tb_offset=None) -> None:
        print(f"\x1b[1;31m❌ {valor}\x1b[0m", file=sys.stderr)

    shell.set_custom_exc((ErrorUsuario,), mostrar)


def version_instalada(paquete: str) -> str | None:
    """Versión instalada de un paquete, o None si no está."""
    try:
        return metadata.version(paquete)
    except metadata.PackageNotFoundError:
        return None


def version_mayor_menor(version: str) -> tuple[int, int]:
    """'2.10.0+cu126' → (2, 10)."""
    coincidencia = re.match(r"(\d+)\.(\d+)", version)
    if coincidencia is None:
        raise ValueError(f"Versión no reconocida: {version}")
    return int(coincidencia.group(1)), int(coincidencia.group(2))


def verificar_gpu() -> str:
    """Nombre y memoria de la GPU; error claro si el entorno no tiene una."""
    if shutil.which("nvidia-smi") is None:
        raise ErrorUsuario(
            "Este entorno no tiene GPU. Ve a «Entorno de ejecución → Cambiar tipo de entorno de ejecución», "
            "elige «T4 GPU», guarda y ejecuta otra vez desde la celda 1."
        )
    consulta = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    lineas = consulta.stdout.strip().splitlines()
    if consulta.returncode != 0 or not lineas:
        raise ErrorUsuario("La GPU no responde. Usa «Entorno de ejecución → Reiniciar sesión» y vuelve a intentarlo.")
    return lineas[0].strip()


def asegurar_ffmpeg() -> None:
    """FFmpeg convierte tu audio (MP3, M4A, OPUS, video…). Colab ya lo trae; si faltara, se instala."""
    if shutil.which("ffmpeg") is None:
        subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True, capture_output=True)


def especificacion_torchcodec(torch: tuple[int, int]) -> str | None:
    """torchcodec compatible con la versión de torch, según la tabla oficial de torchcodec.

    coqui-tts lo exige desde torch 2.9, porque torchaudio le delega la lectura de audio.
    """
    if torch < (2, 9):
        return None
    tabla = {(2, 9): "torchcodec==0.9.*", (2, 10): "torchcodec==0.10.*"}
    return tabla.get(torch, "torchcodec>=0.12")  # desde la 0.12 sirve para cualquier torch >= 2.11


def paquetes_pendientes() -> list[str]:
    """Lo que falta instalar para que XTTS-v2 funcione con el torch que ya trae Colab."""
    version_torch = version_instalada("torch")
    if version_torch is None:
        raise ErrorUsuario("Este entorno no tiene PyTorch. Usa un entorno estándar de Colab con GPU.")
    pendientes = []
    if version_instalada("coqui-tts") != VERSION_COQUI_TTS:
        pendientes.append(f"coqui-tts=={VERSION_COQUI_TTS}")
    if version_instalada("torchaudio") is None:
        pendientes.append(f"torchaudio=={version_torch}")  # misma versión y misma variante CUDA que torch
    codec = especificacion_torchcodec(version_mayor_menor(version_torch))
    if codec is not None and version_instalada("torchcodec") is None:
        pendientes.append(codec)
    return pendientes + list(PAQUETES_EXTRA)


def indices_pytorch(especificaciones: list[str]) -> list[str]:
    """Índices de PyTorch para las variantes CUDA pedidas (p. ej. «+cu126»), que no están publicadas en PyPI."""
    variantes = sorted({e.split("+", 1)[1] for e in especificaciones if "+" in e})
    return [f"https://download.pytorch.org/whl/{variante}" for variante in variantes]


def pip_instalar(especificaciones: list[str], indices_extra: list[str]) -> None:
    """pip install silencioso; si falla, muestra el final del registro para diagnosticar."""
    comando = [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check"]
    for indice in indices_extra:
        comando += ["--extra-index-url", indice]
    comando += especificaciones
    resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
    if resultado.returncode != 0:
        print(resultado.stdout[-2000:], resultado.stderr[-4000:], sep="\n")
        raise RuntimeError("pip no pudo instalar: " + " ".join(especificaciones))


def modulos_desactualizados() -> list[str]:
    """Librerías ya cargadas en memoria cuya versión en disco acaba de cambiar."""
    cambios = []
    for nombre in MODULOS_SENSIBLES:
        cargada = getattr(sys.modules.get(nombre), "__version__", None)
        instalada = version_instalada(nombre)
        if cargada and instalada and cargada.split("+")[0] != instalada.split("+")[0]:
            cambios.append(f"{nombre} {cargada} → {instalada}")
    return cambios


def reiniciar_sesion(cambios: list[str]) -> None:
    """Reinicia el kernel para que cargue las versiones nuevas (lo instalado se conserva)."""
    print("🔄 Se actualizaron librerías que ya estaban en memoria:", "; ".join(cambios))
    print("   Reiniciando la sesión… Cuando se reconecte, ejecuta otra vez desde la celda 1 (será inmediata).")
    sys.stdout.flush()
    os.kill(os.getpid(), signal.SIGKILL)


def preparar_entorno() -> None:
    inicio = time.perf_counter()
    registrar_errores_amigables()
    gpu = verificar_gpu()
    asegurar_ffmpeg()
    pendientes = paquetes_pendientes()
    if pendientes:
        print("📦 Instalando:", ", ".join(pendientes), "(1–3 min)…")
        pip_instalar(pendientes, indices_pytorch(pendientes))
        importlib.invalidate_caches()  # que esta misma sesión vea los paquetes recién instalados
    cambios = modulos_desactualizados()
    if cambios:
        reiniciar_sesion(cambios)
    print(f"✅ Entorno listo en {time.perf_counter() - inicio:.0f} s")
    print(
        f"   GPU: {gpu} · Python {sys.version.split()[0]} · "
        f"torch {version_instalada('torch')} · coqui-tts {version_instalada('coqui-tts')}"
    )


# ─── Ejecución ───
preparar_entorno()

In [ ]:
# @title 2a · Funciones: Drive y archivos { display-mode: "form" }
# @markdown Acceso a tu Google Drive, búsqueda de archivos por nombre y nombres de salida. No tiene parámetros: solo ejecútala.

"""Celda 2a · Archivos: Google Drive (montaje, rutas y búsqueda), lectura de .txt y archivos de salida."""

import os
import re
import shutil
import time
import unicodedata
from collections import deque
from pathlib import Path

if "ErrorUsuario" not in globals():
    raise RuntimeError("Ejecuta primero la celda 1 · Preparar el entorno.")


# ═══ Google Drive ═══════════════════════════════════════════════════════════════

RAIZ_DRIVE = Path("/content/drive")
MI_UNIDAD = RAIZ_DRIVE / "MyDrive"
CARPETA_TRABAJO = Path("/content/clonador_voz")
EXTENSIONES_MEDIOS = frozenset({
    ".wav", ".mp3", ".m4a", ".aac", ".ogg", ".oga", ".opus", ".flac", ".wma", ".amr",
    ".webm", ".mp4", ".mov", ".mkv", ".avi", ".3gp",
})
EXTENSIONES_TEXTO = frozenset({".txt", ".md"})
PREFIJOS_MI_UNIDAD = ("/content/drive/MyDrive/", "MyDrive/", "My Drive/", "Mi unidad/")
NOMBRES_MI_UNIDAD = frozenset({"/content/drive/MyDrive", "MyDrive", "My Drive", "Mi unidad"})


INSTRUCCIONES_DRIVE = (
    "No se pudo conectar tu Google Drive. Vuelve a ejecutar la celda y, en la ventana de Google: "
    "1) elige la MISMA cuenta con la que abriste Colab y 2) marca TODOS los permisos («Seleccionar todo») antes de "
    "«Continuar». Si sigue fallando: menú «Entorno de ejecución → Desconectar y eliminar entorno de ejecución» "
    "(Runtime → Disconnect and delete runtime) y empieza desde la celda 1, o conecta Drive con el botón de Drive del "
    "panel «Archivos» (ícono de carpeta, a la izquierda). En la celda 4 también puedes subir el audio desde tu computadora."
)


class DriveNoDisponible(ErrorUsuario):
    """Drive no se pudo montar: quien llama decide si es un error (leer un archivo) o solo un aviso (guardar)."""

    def __init__(self, detalle: str) -> None:
        super().__init__(f"{INSTRUCCIONES_DRIVE} (Detalle técnico: {detalle})")
        self.detalle = detalle


def drive_montado() -> bool:
    """¿Drive está montado de verdad? Una carpeta «MyDrive» creada a mano no cuenta."""
    return os.path.ismount(RAIZ_DRIVE) and MI_UNIDAD.is_dir()


def montar_drive(intentos: int = 2) -> Path:
    """Monta Google Drive (la primera vez pide permiso) y devuelve «Mi unidad».

    «mount failed» suele deberse a permisos incompletos, a otra cuenta o a un fallo pasajero: se reintenta una vez
    y, si persiste, se explica qué hacer.
    """
    if drive_montado():
        return MI_UNIDAD
    from google.colab import drive

    ultimo_error: Exception | None = None
    for intento in range(1, intentos + 1):
        try:
            drive.mount(str(RAIZ_DRIVE), force_remount=intento > 1)
        except Exception as error:  # noqa: BLE001 — ValueError («mount failed»), MessageError («credential propagation…»)
            ultimo_error = error
            if intento < intentos:
                print(f"⚠️ Drive no se conectó ({error}). Reintentando; si aparece la ventana de Google, marca todos los permisos.")
            continue
        if drive_montado():
            return MI_UNIDAD
        ultimo_error = RuntimeError("Drive respondió, pero «Mi unidad» no apareció")
    raise DriveNoDisponible(f"{type(ultimo_error).__name__}: {ultimo_error}")


def clave_nombre(nombre: str) -> str:
    """Forma comparable de un nombre: Unicode NFC y sin mayúsculas.

    Drive conserva tal cual los nombres de archivos subidos desde Mac, cuyas tildes vienen en forma NFD.
    """
    return unicodedata.normalize("NFC", nombre).casefold()


def ruta_en_mi_unidad(texto: str) -> Path:
    """Interpreta lo escrito como ruta dentro de «Mi unidad»; solo una ruta completa de /content sale de ella."""
    limpio = unicodedata.normalize("NFC", texto.strip().strip("\"'").strip())
    if limpio.rstrip("/") in NOMBRES_MI_UNIDAD:
        return MI_UNIDAD
    for prefijo in PREFIJOS_MI_UNIDAD:
        if limpio.startswith(prefijo):
            return MI_UNIDAD / limpio[len(prefijo):]
    if limpio.startswith("/content/"):
        return Path(limpio)  # p. ej. una unidad compartida: /content/drive/Shareddrives/…
    return MI_UNIDAD / limpio.lstrip("/")  # «/Carpeta» también es relativa a Mi unidad


def describir_ruta(ruta: Path) -> str:
    """Ruta legible: «Mi unidad/…» para Drive y la ruta completa en otro caso."""
    try:
        return f"Mi unidad/{ruta.relative_to(MI_UNIDAD)}"
    except ValueError:
        return str(ruta)


def resolver_en_drive(entrada: str, *, extensiones: frozenset[str], incluir_carpetas: bool) -> Path:
    """La ruta indicada (tolera mayúsculas, formas Unicode y extensión omitida) o, si es solo un nombre, lo busca."""
    limpio = entrada.strip().strip("\"'").strip()
    if not limpio:
        raise ErrorUsuario("Falta el nombre del archivo en Drive.")
    ruta = ruta_en_mi_unidad(limpio)
    if ruta.exists():
        return ruta
    if "/" not in limpio:
        return buscar_por_nombre(limpio, MI_UNIDAD, extensiones=extensiones, incluir_carpetas=incluir_carpetas)
    encontrada = resolver_por_partes(ruta, extensiones)
    if encontrada is None:
        raise ErrorUsuario(f"No existe «{limpio}» en Mi unidad. Revisa la ruta y el nombre del archivo.")
    print(f"📄 Usando {describir_ruta(encontrada)}")
    return encontrada


def resolver_por_partes(ruta: Path, extensiones: frozenset[str]) -> Path | None:
    """Recorre la ruta tramo a tramo: el tramo que no existe tal cual se busca entre sus hermanos por su clave."""
    actual = Path(ruta.anchor)
    partes = ruta.parts[1:]
    for indice, parte in enumerate(partes):
        exacta = actual / parte
        if exacta.exists():
            actual = exacta
            continue
        objetivo, ultimo = clave_nombre(parte), indice == len(partes) - 1
        try:
            hermanos = sorted(os.scandir(actual), key=lambda entrada: entrada.name)
        except OSError:
            return None
        coincidencia = next(
            (
                h for h in hermanos
                if clave_nombre(h.name) == objetivo or (ultimo and h.is_file() and _coincide(h.name, objetivo, extensiones))
            ),
            None,
        )
        if coincidencia is None:
            return None
        actual = Path(coincidencia.path)
    return actual


def _coincide(nombre: str, objetivo: str, extensiones: frozenset[str]) -> bool:
    """Igualdad por clave de nombre; si se omitió la extensión, acepta cualquiera de las válidas."""
    nombre = clave_nombre(nombre)
    if nombre == objetivo:
        return True
    raiz, extension = os.path.splitext(nombre)
    return raiz == objetivo and extension in extensiones


def _fecha_modificacion(ruta: Path) -> float:
    try:
        return ruta.stat().st_mtime
    except OSError:
        return 0.0


def buscar_por_nombre(
    nombre: str,
    raiz: Path,
    *,
    extensiones: frozenset[str],
    incluir_carpetas: bool,
    limite_s: float = 60.0,
    margen_s: float = 4.0,
) -> Path:
    """Búsqueda en anchura (primero lo menos profundo) con tiempo límite; ante duplicados, el más reciente."""
    objetivo = clave_nombre(nombre)
    print(f"🔎 Buscando «{nombre}» en tu Drive…")
    encontrados: list[Path] = []
    pendientes: deque[Path] = deque([raiz])
    fin = time.monotonic() + limite_s
    while pendientes and time.monotonic() < fin:
        try:
            entradas = list(os.scandir(pendientes.popleft()))
        except OSError:
            continue
        for entrada in entradas:
            if entrada.name.startswith("."):
                continue
            if entrada.is_dir(follow_symlinks=False):
                pendientes.append(Path(entrada.path))
                if incluir_carpetas and clave_nombre(entrada.name) == objetivo:
                    encontrados.append(Path(entrada.path))
            elif _coincide(entrada.name, objetivo, extensiones):
                encontrados.append(Path(entrada.path))
        if encontrados:  # margen breve para detectar homónimos y terminar
            fin = min(fin, time.monotonic() + margen_s)
    if not encontrados:
        raise ErrorUsuario(f"No encontré «{nombre}» en Mi unidad. Escribe su ruta, p. ej. «Carpeta/{nombre}».")
    elegido = max(encontrados, key=_fecha_modificacion)
    if len(encontrados) > 1:
        print(f"ℹ️ Hay {len(encontrados)} coincidencias; uso la más reciente. Para usar otra, escribe su ruta:")
        for ruta in encontrados[:5]:
            print("   •", describir_ruta(ruta))
    print(f"📄 Usando {describir_ruta(elegido)}")
    return elegido


def listar_medios(ruta: Path, maximo: int = 12) -> list[Path]:
    """El archivo indicado, o los audios y videos de una carpeta (sin subcarpetas) en orden alfabético."""
    if ruta.is_file():
        return [ruta]
    archivos = sorted(
        (p for p in ruta.iterdir() if p.is_file() and p.suffix.casefold() in EXTENSIONES_MEDIOS),
        key=lambda p: p.name.casefold(),
    )
    if not archivos:
        raise ErrorUsuario(f"La carpeta «{ruta.name}» no tiene audios ni videos.")
    if len(archivos) > maximo:
        print(f"ℹ️ La carpeta tiene {len(archivos)} archivos; uso los primeros {maximo}.")
    return archivos[:maximo]


def leer_texto_de_archivo(ruta: Path) -> str:
    """UTF-8 (con o sin BOM); si no lo es, Windows-1252 (Bloc de notas antiguo) y, como último recurso, Latin-1."""
    datos = ruta.read_bytes()
    for codificacion in ("utf-8-sig", "cp1252"):
        try:
            return datos.decode(codificacion)
        except UnicodeDecodeError:
            continue
    return datos.decode("latin-1")


# ═══ Archivos de salida ═════════════════════════════════════════════════════════


def preparar_carpeta(ruta: Path, *, vaciar: bool = False) -> Path:
    """Crea la carpeta (y sus padres); con vaciar=True la deja vacía antes."""
    if vaciar and ruta.exists():
        shutil.rmtree(ruta)
    ruta.mkdir(parents=True, exist_ok=True)
    return ruta


def nombre_seguro(nombre: str, por_defecto: str = "voz_clonada") -> str:
    """Nombre de archivo válido: sin extensión de audio ni caracteres prohibidos."""
    base = re.sub(r"\.(wav|mp3)$", "", nombre.strip(), flags=re.IGNORECASE)
    base = re.sub(r'[\\/:*?"<>|\x00-\x1f]+', "_", base).strip(" ._")
    return base[:120] or por_defecto


def ruta_libre(carpeta: Path, base: str, extension: str) -> Path:
    """Primera ruta que no sobrescribe nada: nombre.ext, nombre_2.ext, nombre_3.ext…"""
    ruta, n = carpeta / f"{base}{extension}", 2
    while ruta.exists():
        ruta, n = carpeta / f"{base}_{n}{extension}", n + 1
    return ruta


def formatear_duracion(segundos: float) -> str:
    minutos, resto = divmod(segundos, 60.0)
    return f"{int(minutos)} min {resto:.0f} s" if minutos >= 1 else f"{resto:.1f} s"


# ─── Ejecución ───
print("✅ Funciones de archivos cargadas.")

In [ ]:
# @title 2b · Funciones: audio y volumen { display-mode: "form" }
# @markdown Conversión con FFmpeg, procesamiento del audio con NumPy y nivelado de volumen (LUFS). No tiene parámetros: solo ejecútala.

"""Celda 2b · Audio: FFmpeg, WAV y procesamiento con NumPy, sonoridad ITU-R BS.1770-4 con limitador y reproductor."""

import subprocess
import wave
from collections.abc import Sequence
from pathlib import Path

import numpy as np

if "preparar_carpeta" not in globals():
    raise RuntimeError("Ejecuta primero las celdas 1 y 2a.")


# ═══ Audio con FFmpeg ═══════════════════════════════════════════════════════════

FRECUENCIA_REFERENCIA = 22050  # Hz con los que XTTS analiza la voz de referencia


def filtros_referencia(limpiar: bool, reducir_ruido: bool) -> str:
    """Cadena de filtros de FFmpeg para preparar la voz de referencia."""
    filtros: list[str] = []
    if limpiar:
        filtros.append("highpass=f=70")  # retumbes y ruido grave
    if reducir_ruido:
        filtros.append("afftdn=nr=10:nf=-30")  # reducción espectral moderada
    if limpiar:
        recorte = "silenceremove=start_periods=1:start_duration=0.05:start_threshold=-45dB"
        filtros += [recorte, "areverse", recorte, "areverse"]  # silencio inicial y final
        filtros.append("loudnorm=I=-20:TP=-2:LRA=11")  # volumen homogéneo (EBU R128)
    return ",".join(filtros)


def ejecutar_ffmpeg(argumentos: Sequence[str], tarea: str) -> None:
    """Ejecuta FFmpeg sin ruido en pantalla; si falla, explica qué no pudo hacer."""
    comando = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", *argumentos]
    resultado = subprocess.run(comando, capture_output=True, text=True, check=False)
    if resultado.returncode != 0:
        detalle = (resultado.stderr.strip().splitlines() or ["sin detalle"])[-1]
        raise ErrorUsuario(f"FFmpeg no pudo {tarea}: {detalle}")


def preparar_referencia(origen: Path, destino: Path, *, filtros: str, max_segundos: float) -> None:
    """Cualquier audio o video → WAV mono de 16 bits a 22 050 Hz, filtrado y recortado."""
    argumentos = ["-t", f"{max_segundos + 30:.0f}", "-i", str(origen), "-vn", "-ac", "1"]
    if filtros:
        argumentos += ["-af", filtros]
    argumentos += ["-ar", str(FRECUENCIA_REFERENCIA), "-t", f"{max_segundos:.2f}", "-c:a", "pcm_s16le", str(destino)]
    ejecutar_ffmpeg(argumentos, f"leer «{origen.name}» como audio")


def convertir_a_mp3(origen: Path, destino: Path, kbps: int = 192) -> None:
    """WAV → MP3 a 44,1 kHz: a los 24 kHz del modelo, el MP3 no admite 192 kbps y bajaría a 160 sin avisar."""
    argumentos = ["-i", str(origen), "-ar", "44100", "-c:a", "libmp3lame", "-b:a", f"{kbps}k", str(destino)]
    ejecutar_ffmpeg(argumentos, "crear el MP3")


# ═══ WAV y procesamiento con NumPy ══════════════════════════════════════════════


def leer_wav(ruta: Path) -> tuple[np.ndarray, int]:
    """WAV PCM de 16 bits → (muestras float32 en [-1, 1], frecuencia). Sin librerías de audio."""
    with wave.open(str(ruta), "rb") as wav:
        canales, ancho, frecuencia = wav.getnchannels(), wav.getsampwidth(), wav.getframerate()
        datos = wav.readframes(wav.getnframes())
    if ancho != 2:
        raise ValueError(f"{ruta.name}: se esperaba PCM de 16 bits")
    muestras = np.frombuffer(datos, dtype="<i2").astype(np.float32) / np.float32(32768.0)
    if canales > 1:
        muestras = muestras.reshape(-1, canales).mean(axis=1, dtype=np.float32)
    return muestras, frecuencia


def escribir_wav(ruta: Path, muestras: np.ndarray, frecuencia: int) -> None:
    """float32 en [-1, 1] → WAV mono PCM de 16 bits."""
    pcm = np.round(np.clip(muestras, -1.0, 1.0) * 32767.0).astype("<i2")
    with wave.open(str(ruta), "wb") as wav:
        wav.setnchannels(1)
        wav.setsampwidth(2)
        wav.setframerate(frecuencia)
        wav.writeframes(pcm.tobytes())


def rms_por_tramas(x: np.ndarray, ventana: int, salto: int) -> np.ndarray:
    """RMS de tramas solapadas en O(n) con una suma acumulada de cuadrados (sin copiar tramas)."""
    if x.size == 0:
        return np.zeros(0)
    if x.size <= ventana:
        return np.sqrt([np.mean(np.square(x, dtype=np.float64))])
    acumulada = np.concatenate(([0.0], np.cumsum(np.square(x, dtype=np.float64))))
    inicios = np.arange(0, x.size - ventana + 1, salto)
    energia = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    return np.sqrt(np.maximum(energia, 0.0))


def recortar_silencios(x: np.ndarray, frecuencia: int, umbral_db: float = -45.0, margen_ms: float = 60.0) -> np.ndarray:
    """Quita el silencio inicial y final: conserva de la primera a la última trama con voz."""
    ventana, salto = max(1, frecuencia // 50), max(1, frecuencia // 100)  # tramas de 20 ms cada 10 ms
    rms = rms_por_tramas(x, ventana, salto)
    if rms.size == 0 or rms.max() < 1e-5:
        return x[:0]
    activas = np.flatnonzero(rms >= rms.max() * 10.0 ** (umbral_db / 20.0))
    margen = int(frecuencia * margen_ms / 1000.0)
    inicio = max(0, int(activas[0]) * salto - margen)
    fin = min(x.size, int(activas[-1]) * salto + ventana + margen)
    return x[inicio:fin]


def aplicar_fundidos(x: np.ndarray, frecuencia: int, ms: float = 8.0) -> np.ndarray:
    """Fundido de entrada y de salida muy corto para que las uniones no hagan «clic»."""
    n = min(int(frecuencia * ms / 1000.0), x.size // 2)
    if n == 0:
        return x
    y = x.astype(np.float32, copy=True)
    rampa = np.linspace(0.0, 1.0, n, dtype=np.float32)
    y[:n] *= rampa
    y[-n:] *= rampa[::-1]
    return y


def normalizar_pico(x: np.ndarray, dbfs: float = -1.0) -> np.ndarray:
    """Escala la señal para que su pico quede en `dbfs` (−1 dBFS deja margen al MP3)."""
    pico = float(np.max(np.abs(x))) if x.size else 0.0
    if pico < 1e-6:
        return x.astype(np.float32, copy=False)
    return (x * np.float32(10.0 ** (dbfs / 20.0) / pico)).astype(np.float32, copy=False)


def concatenar_con_pausas(fragmentos: Sequence[np.ndarray], pausas_s: Sequence[float], frecuencia: int) -> np.ndarray:
    """Une los fragmentos dejando tras cada uno su pausa; reserva la memoria una sola vez."""
    pausas = [max(0, round(p * frecuencia)) for p in pausas_s]
    salida = np.zeros(sum(f.size for f in fragmentos) + sum(pausas), dtype=np.float32)
    posicion = 0
    for fragmento, pausa in zip(fragmentos, pausas, strict=True):
        salida[posicion : posicion + fragmento.size] = fragmento
        posicion += fragmento.size + pausa
    return salida


# ═══ Volumen: sonoridad ITU-R BS.1770-4 (EBU R128) y limitador ══════════════════

GANANCIA_MAXIMA_DB = 30.0  # no se amplifica más que esto aunque el audio venga muy bajo


def coeficientes_filtro_k(frecuencia: int) -> tuple[tuple[np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray]]:
    """(b, a) de las dos etapas del filtro K (estante de agudos + pasa altos) para cualquier frecuencia de muestreo.

    Fórmulas de libebur128; a 48 kHz reproducen los coeficientes tabulados en la norma.
    """
    f0, ganancia_db, q = 1681.974450955533, 3.999843853973347, 0.7071752369554196
    k = np.tan(np.pi * f0 / frecuencia)
    vh = 10.0 ** (ganancia_db / 20.0)
    vb = vh**0.4996667741545416
    a0 = 1.0 + k / q + k * k
    estante = (
        np.array([(vh + vb * k / q + k * k) / a0, 2.0 * (k * k - vh) / a0, (vh - vb * k / q + k * k) / a0]),
        np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]),
    )
    f0, q = 38.13547087602444, 0.5003270373238773
    k = np.tan(np.pi * f0 / frecuencia)
    a0 = 1.0 + k / q + k * k
    pasa_altos = (np.array([1.0, -2.0, 1.0]), np.array([1.0, 2.0 * (k * k - 1.0) / a0, (1.0 - k / q + k * k) / a0]))
    return estante, pasa_altos


def largo_fft_rapido(n: int) -> int:
    """Menor 2^a·3^b·5^c ≥ n: con esos tamaños la FFT es rápida y casi no se rellena de más."""
    mejor = 1 << max(0, n - 1).bit_length()
    potencia5 = 1
    while potencia5 < mejor:
        potencia35 = potencia5
        while potencia35 < mejor:
            veces = -(-n // potencia35)  # cuántas veces cabe: se completa con la menor potencia de 2 suficiente
            mejor = min(mejor, potencia35 << (veces - 1).bit_length())
            potencia35 *= 3
        potencia5 *= 5
    return mejor


def ponderar_k(x: np.ndarray, frecuencia: int) -> np.ndarray:
    """Aplica el filtro K multiplicando espectros: con 0,5 s de relleno equivale al filtro IIR, sin bucles en Python."""
    largo = largo_fft_rapido(x.size + frecuencia // 2)
    z = np.exp(-2j * np.pi * np.arange(largo // 2 + 1) / largo)  # z⁻¹ en cada frecuencia de la FFT
    respuesta = np.ones_like(z)
    for b, a in coeficientes_filtro_k(frecuencia):
        respuesta *= (b[0] + z * (b[1] + z * b[2])) / (a[0] + z * (a[1] + z * a[2]))
    return np.fft.irfft(np.fft.rfft(x, largo) * respuesta, largo)[: x.size]


def sonoridad_integrada(x: np.ndarray, frecuencia: int) -> float:
    """Sonoridad integrada (LUFS) de una señal mono: bloques de 400 ms con 75 % de solapamiento,
    compuerta absoluta de −70 LUFS y relativa de −10 LU. Devuelve −inf si todo es silencio."""
    y = ponderar_k(np.asarray(x, dtype=np.float64), frecuencia)
    ventana, salto = round(0.4 * frecuencia), round(0.1 * frecuencia)
    acumulada = np.concatenate(([0.0], np.cumsum(y * y)))
    if y.size >= ventana:
        inicios = np.arange(0, y.size - ventana + 1, salto)
        energias = (acumulada[inicios + ventana] - acumulada[inicios]) / ventana
    else:
        energias = np.array([acumulada[-1] / max(1, y.size)])
    energias = np.maximum(energias, 1e-20)
    sonoridades = -0.691 + 10.0 * np.log10(energias)
    energias, sonoridades = energias[sonoridades > -70.0], sonoridades[sonoridades > -70.0]
    if energias.size == 0:
        return float("-inf")
    compuerta = -0.691 + 10.0 * np.log10(energias.mean()) - 10.0
    return float(-0.691 + 10.0 * np.log10(energias[sonoridades > compuerta].mean()))


def minimo_deslizante(x: np.ndarray, ventana: int) -> np.ndarray:
    """m[i] = min(x[i : i + ventana]) en O(n) con el algoritmo de van Herk/Gil-Werman, todo vectorizado:
    cada ventana cae en dos bloques consecutivos, y basta el mínimo-sufijo de uno y el mínimo-prefijo del otro."""
    if ventana <= 1 or x.size == 0:
        return x.astype(np.float64, copy=True)
    bloques = -(-(x.size + ventana - 1) // ventana)
    relleno = np.full(bloques * ventana, np.inf)
    relleno[: x.size] = x
    matriz = relleno.reshape(bloques, ventana)
    prefijo = np.minimum.accumulate(matriz, axis=1).ravel()
    sufijo = np.minimum.accumulate(matriz[:, ::-1], axis=1)[:, ::-1].ravel()
    indices = np.arange(x.size)
    return np.minimum(sufijo[indices], prefijo[indices + ventana - 1])


def limitar_picos(x: np.ndarray, frecuencia: int, techo_dbfs: float = -1.0) -> np.ndarray:
    """Limitador con anticipación: la ganancia baja en rampa antes de cada pico y ninguna muestra pasa el techo.

    g[n] = media de los últimos 7,5 ms de min(ganancia necesaria en los próximos 15 ms): como la media abarca
    menos que el mínimo, en cada pico la ganancia nunca supera la necesaria.
    """
    techo = 10.0 ** (techo_dbfs / 20.0)
    amplitud = np.abs(np.asarray(x, dtype=np.float64))
    if amplitud.size == 0 or amplitud.max() <= techo:
        return np.asarray(x, dtype=np.float32)
    anticipacion, suavizado = max(1, round(0.015 * frecuencia)), max(1, round(0.0075 * frecuencia))
    envolvente = minimo_deslizante(np.minimum(1.0, techo / np.maximum(amplitud, 1e-12)), anticipacion)
    acumulada = np.concatenate(([0.0], np.cumsum(envolvente)))
    fin = np.arange(1, amplitud.size + 1)
    inicio = np.maximum(0, fin - suavizado)
    ganancia = (acumulada[fin] - acumulada[inicio]) / (fin - inicio)
    return np.clip(x * ganancia, -techo, techo).astype(np.float32)  # el recorte solo absorbe redondeos


def nivelar_volumen(x: np.ndarray, frecuencia: int, objetivo_lufs: float, techo_dbfs: float = -1.0) -> np.ndarray:
    """Lleva la sonoridad integrada al objetivo (volumen percibido parejo) y limita los picos al techo."""
    actual = sonoridad_integrada(x, frecuencia)
    if not np.isfinite(actual):
        return normalizar_pico(x, techo_dbfs)
    ganancia_db = min(objetivo_lufs - actual, GANANCIA_MAXIMA_DB)
    return limitar_picos(np.asarray(x, dtype=np.float64) * 10.0 ** (ganancia_db / 20.0), frecuencia, techo_dbfs)


LIMITE_REPRODUCTOR_MB = 8.0  # el reproductor incrusta el audio en el cuaderno: más grande lo vuelve pesado
SEGUNDOS_VISTA_PREVIA = 60


def mostrar_reproductor(ruta: Path) -> None:
    """Reproductor dentro de la celda; si el archivo es grande, solo suena su primer minuto."""
    from IPython.display import Audio, display

    if ruta.stat().st_size > LIMITE_REPRODUCTOR_MB * 1_000_000:
        vista = preparar_carpeta(CARPETA_TRABAJO / "vistas_previas") / f"{ruta.stem}.mp3"
        argumentos = ["-t", str(SEGUNDOS_VISTA_PREVIA), "-i", str(ruta), "-c:a", "libmp3lame", "-b:a", "96k", str(vista)]
        ejecutar_ffmpeg(argumentos, "crear la vista previa")
        print("ℹ️ Audio largo: aquí suena el primer minuto; el archivo completo está guardado.")
        ruta = vista
    display(Audio(filename=str(ruta)))


# ─── Ejecución ───
print("✅ Funciones de audio cargadas.")

In [ ]:
# @title 2c · Funciones: texto, idiomas y pronunciación { display-mode: "form" }
# @markdown División en frases y pausas, idiomas mezclados, montos y cifras en español y reglas de pronunciación. No tiene parámetros: solo ejecútala.

"""Celda 2c · Texto: frases y pausas, tramos por idioma, montos y cifras en español y diccionario de pronunciación."""

import re
import unicodedata
from collections.abc import Sequence
from dataclasses import dataclass


# ═══ Texto ══════════════════════════════════════════════════════════════════════


@dataclass(frozen=True)
class Segmento:
    """Trozo de texto que el modelo lee de una vez, la pausa (en segundos) que va después y su idioma."""

    texto: str
    pausa: float
    idioma: str = ""  # vacío: el idioma principal elegido en la celda 5


PATRON_PAUSA = re.compile(r"\[\s*pausa(?:\s*[=:]?\s*(\d+(?:[.,]\d+)?)\s*s?)?\s*\]", re.IGNORECASE)
PATRON_FIN_FRASE = re.compile(r"(?<=[.!?…।؟])\s+")
PATRON_CORTE_SUAVE = re.compile(r"(?<=[,;:—–])\s+")
PAUSA_MARCA = 1.0  # segundos de un [pausa] sin número
MINIMO_CARACTERES = 20  # frases más cortas se unen a la siguiente: XTTS falla con textos muy breves
# Un punto tras estas palabras (o tras una sola letra: «p. ej.», «J. K. Rowling») no cierra la frase.
ABREVIATURAS = frozenset({
    "sr", "sra", "srta", "sres", "dr", "dra", "dres", "ing", "lic", "mg", "mag", "mtro", "mtra", "prof", "profa",
    "arq", "abg", "abog", "gral", "cnel", "sto", "sta", "mons", "pdte", "ud", "uds", "mr", "mrs", "ms", "st", "jr",
    "av", "avda", "ca", "cdra", "mz", "lt", "urb", "dpto", "depto", "psje", "nro", "núm", "num", "pág", "págs", "pag",
    "fig", "vol", "ej", "aprox", "máx", "mín", "tel", "cel", "cía", "inc", "vs", "e.g", "i.e", "s/",
})


def pausa_de_parrafo(pausa_frase: float) -> float:
    return max(0.6, 2.0 * pausa_frase)


def dividir_texto(texto: str, limite: int, pausa_frase: float) -> list[Segmento]:
    """Texto → segmentos que el modelo puede leer, con la pausa que va tras cada uno.

    [pausa] o [pausa 2.5] fija un silencio; cada salto de línea es un párrafo (pausa más larga).
    """
    pausa_parrafo = pausa_de_parrafo(pausa_frase)
    segmentos: list[Segmento] = []
    for bloque, pausa_marca in _bloques_por_marcas(texto):
        parrafos = [p for p in re.split(r"\n+", bloque) if p.strip()]
        for i, parrafo in enumerate(parrafos):
            frases = _frases(parrafo, limite)
            for j, frase in enumerate(frases):
                if j < len(frases) - 1:
                    pausa = pausa_frase
                else:
                    pausa = pausa_parrafo if i < len(parrafos) - 1 else 0.0
                segmentos.append(Segmento(frase, pausa))
        if segmentos and pausa_marca:
            segmentos[-1] = Segmento(segmentos[-1].texto, segmentos[-1].pausa + pausa_marca)
    return segmentos


def _bloques_por_marcas(texto: str) -> list[tuple[str, float]]:
    """Corta el texto en cada marca de pausa; devuelve (bloque, segundos de la marca que lo sigue)."""
    bloques, inicio = [], 0
    for marca in PATRON_PAUSA.finditer(texto):
        segundos = float(marca.group(1).replace(",", ".")) if marca.group(1) else PAUSA_MARCA
        bloques.append((texto[inicio : marca.start()], segundos))
        inicio = marca.end()
    bloques.append((texto[inicio:], 0.0))
    return bloques


def pausa_inicial(texto: str) -> float:
    """Segundos de las marcas [pausa] que aparecen antes de la primera palabra (todas, si no hay palabras)."""
    total = 0.0
    for bloque, segundos in _bloques_por_marcas(texto):
        if any(caracter.isalnum() for caracter in bloque):
            break
        total += segundos
    return total


def _frases(parrafo: str, limite: int) -> list[str]:
    """Frases de un párrafo: corta en . ! ? …, parte las demasiado largas y une las muy cortas."""
    limpio = re.sub(r"\s+", " ", parrafo).strip()
    piezas: list[str] = []
    for frase in _dividir_en_frases(limpio):
        piezas.extend(_partir_frase_larga(frase, limite))
    return [frase for frase in _unir_cortas(piezas, limite) if any(c.isalnum() for c in frase)]  # nada de «.» sueltos


def _dividir_en_frases(texto: str) -> list[str]:
    """Corta tras . ! ? … salvo cuando el punto cierra una abreviatura («Dr. Pérez», «Av. Grau», «S/. 150»)."""
    frases: list[str] = []
    inicio = 0
    for corte in PATRON_FIN_FRASE.finditer(texto):
        if _termina_en_abreviatura(texto[inicio : corte.start()]):
            continue
        frases.append(texto[inicio : corte.start()])
        inicio = corte.end()
    frases.append(texto[inicio:])
    return [frase for frase in frases if frase]


def _termina_en_abreviatura(texto: str) -> bool:
    """¿El punto final de `texto` cierra una abreviatura y no una frase?"""
    if not texto.endswith("."):
        return False
    palabra = texto.rsplit(maxsplit=1)[-1].lstrip("([«“\"'¿¡").rstrip(".").casefold()
    return (len(palabra) == 1 and palabra.isalpha()) or palabra in ABREVIATURAS


def _partir_frase_larga(frase: str, limite: int) -> list[str]:
    """Una frase que excede el límite se parte primero en , ; : — y, si no basta, entre palabras."""
    if len(frase) <= limite:
        return [frase]
    trozos: list[str] = []
    for pieza in PATRON_CORTE_SUAVE.split(frase):
        trozos.extend(_partir_por_palabras(pieza, limite) if len(pieza) > limite else [pieza])
    return _empaquetar(trozos, limite)


def _partir_por_palabras(texto: str, limite: int) -> list[str]:
    palabras: list[str] = []
    for palabra in texto.split():
        while len(palabra) > limite:  # una «palabra» enorme (URL, etc.) se corta en seco
            palabras.append(palabra[:limite])
            palabra = palabra[limite:]
        palabras.append(palabra)
    return _empaquetar(palabras, limite)


def _empaquetar(piezas: Sequence[str], limite: int) -> list[str]:
    """Une piezas consecutivas con espacios sin pasar del límite (voraz, respeta el orden)."""
    grupos: list[str] = []
    actual = ""
    for pieza in piezas:
        candidato = f"{actual} {pieza}" if actual else pieza
        if len(candidato) <= limite:
            actual = candidato
        else:
            if actual:
                grupos.append(actual)
            actual = pieza
    if actual:
        grupos.append(actual)
    return grupos


def _unir_cortas(frases: Sequence[str], limite: int) -> list[str]:
    """Une las frases de menos de MINIMO_CARACTERES con la siguiente (o con la anterior si es la última)."""
    resultado: list[str] = []
    pendiente = ""
    for frase in frases:
        if pendiente and len(pendiente) + 1 + len(frase) > limite:
            resultado.append(pendiente)
            pendiente = ""
        actual = f"{pendiente} {frase}" if pendiente else frase
        if len(actual) < MINIMO_CARACTERES:
            pendiente = actual
        else:
            resultado.append(actual)
            pendiente = ""
    if pendiente:
        if resultado and len(resultado[-1]) + 1 + len(pendiente) <= limite:
            resultado[-1] = f"{resultado[-1]} {pendiente}"
        else:
            resultado.append(pendiente)
    return resultado


def partir_por_la_mitad(texto: str) -> tuple[str, str] | None:
    """Corta en el espacio más cercano al centro; sirve para reintentar un fragmento que el modelo rechazó."""
    espacios = [i for i, caracter in enumerate(texto) if caracter == " "]
    if not espacios:
        return None
    corte = min(espacios, key=lambda i: abs(i - len(texto) // 2))
    return texto[:corte].strip(), texto[corte + 1 :].strip()


# ═══ Idiomas mezclados ══════════════════════════════════════════════════════════

PATRON_ETIQUETA_IDIOMA = re.compile(r"\[(/?)([a-z]{2}(?:-[a-z]{2})?)\]", re.IGNORECASE)
PUNTUACION_INICIAL = re.compile(r"^\s*[.,;:!?…»”\"')\]]+")


@dataclass(frozen=True)
class Tramo:
    """Parte continua del texto que se lee en un mismo idioma."""

    texto: str
    idioma: str


def dividir_por_idioma(texto: str, principal: str, disponibles: frozenset[str]) -> tuple[list[Tramo], list[str]]:
    """«Hola [en]Hello![/en] amigo» → tramos (es, en, es). `[xx]` pasa al idioma xx y `[/xx]` vuelve al principal.

    Una etiqueta sin cierre vale hasta el final; un código no disponible se avisa y se lee en el idioma principal.
    """
    tramos: list[Tramo] = []
    avisos: list[str] = []
    actual, inicio = principal, 0
    for etiqueta in PATRON_ETIQUETA_IDIOMA.finditer(texto):
        _agregar_tramo(tramos, texto[inicio : etiqueta.start()], actual)
        inicio = etiqueta.end()
        cierre, codigo = etiqueta.group(1) == "/", etiqueta.group(2).lower()
        if cierre or codigo in disponibles:
            actual = principal if cierre else codigo
        else:
            avisos.append(f"Idioma «{codigo}» no disponible ({etiqueta.group(0)}): esa parte se lee en el idioma principal.")
            actual = principal
    _agregar_tramo(tramos, texto[inicio:], actual)
    return tramos, avisos


def _agregar_tramo(tramos: list[Tramo], texto: str, idioma: str) -> None:
    """Agrega un tramo. La puntuación que sigue a un cierre («[/en].») y los espacios sueltos van al tramo anterior,
    para no perder la entonación final ni los saltos de párrafo."""
    if tramos:
        puntuacion = PUNTUACION_INICIAL.match(texto)
        if puntuacion:
            tramos[-1] = Tramo(tramos[-1].texto + puntuacion.group(0).strip(), tramos[-1].idioma)
            texto = texto[puntuacion.end() :]
        if not texto.strip() or tramos[-1].idioma == idioma:
            tramos[-1] = Tramo(tramos[-1].texto + texto, tramos[-1].idioma)
            return
    if texto.strip():
        tramos.append(Tramo(texto, idioma))


def hay_salto_de_parrafo(anterior: str, siguiente: str) -> bool:
    """¿Hay un salto de línea entre el final de un tramo y el inicio del siguiente?"""
    return "\n" in anterior[len(anterior.rstrip()) :] + siguiente[: len(siguiente) - len(siguiente.lstrip())]


# ═══ Montos y cifras en español ═════════════════════════════════════════════════
# XTTS no reconoce «S/» y, en español, toma la coma como decimal: lee «1,500» como «uno punto cinco».

_UNIDADES = (
    "cero", "uno", "dos", "tres", "cuatro", "cinco", "seis", "siete", "ocho", "nueve", "diez",
    "once", "doce", "trece", "catorce", "quince", "dieciséis", "diecisiete", "dieciocho", "diecinueve",
    "veinte", "veintiuno", "veintidós", "veintitrés", "veinticuatro", "veinticinco", "veintiséis",
    "veintisiete", "veintiocho", "veintinueve",
)
_DECENAS = ("", "", "", "treinta", "cuarenta", "cincuenta", "sesenta", "setenta", "ochenta", "noventa")
_CENTENAS = (
    "", "ciento", "doscientos", "trescientos", "cuatrocientos", "quinientos",
    "seiscientos", "setecientos", "ochocientos", "novecientos",
)
_ESCALAS = (("", ""), ("millón", "millones"), ("billón", "billones"), ("trillón", "trillones"))


def numero_en_palabras(n: int, *, apocope: bool = False) -> str:
    """Entero ≥ 0 en palabras (escala larga). Con `apocope`, el «uno» final pasa a «un» («veintiún soles»)."""
    if n < 0 or n >= 1000 ** (2 * len(_ESCALAS)):
        raise ValueError(f"Número fuera de rango: {n}")
    if n == 0:
        return "cero"
    grupos: list[str] = []
    for nivel in range(len(_ESCALAS)):
        n, grupo = divmod(n, 1_000_000)
        if grupo:
            grupos.append(_grupo_de_millon(grupo, nivel, apocope and nivel == 0))
        if n == 0:
            break
    return " ".join(reversed(grupos))


def _grupo_de_millon(grupo: int, nivel: int, apocope: bool) -> str:
    """Un grupo de 1 a 999 999 seguido de su escala («millones», «billones»…)."""
    if nivel == 0:
        return _hasta_un_millon(grupo, apocope)
    singular, plural = _ESCALAS[nivel]
    return f"un {singular}" if grupo == 1 else f"{_hasta_un_millon(grupo, True)} {plural}"


def _hasta_un_millon(n: int, apocope: bool) -> str:
    miles, resto = divmod(n, 1000)
    partes: list[str] = []
    if miles == 1:
        partes.append("mil")
    elif miles:
        partes.append(f"{_apocopar(_hasta_mil(miles))} mil")
    if resto:
        palabras = _hasta_mil(resto)
        partes.append(_apocopar(palabras) if apocope else palabras)
    return " ".join(partes)


def _hasta_mil(n: int) -> str:
    """1 a 999."""
    if n == 100:
        return "cien"
    centenas, resto = divmod(n, 100)
    partes = [_CENTENAS[centenas]] if centenas else []
    if resto >= 30:
        decenas, unidades = divmod(resto, 10)
        partes.append(_DECENAS[decenas] + (f" y {_UNIDADES[unidades]}" if unidades else ""))
    elif resto:
        partes.append(_UNIDADES[resto])
    return " ".join(partes)


def _apocopar(palabras: str) -> str:
    """«uno» → «un» y «veintiuno» → «veintiún», como exige el español ante «mil», «millones» o un masculino."""
    if palabras.endswith("veintiuno"):
        return palabras[: -len("veintiuno")] + "veintiún"
    if palabras == "uno" or palabras.endswith(" uno"):
        return palabras[: -len("uno")] + "un"
    return palabras


@dataclass(frozen=True)
class Moneda:
    singular: str
    plural: str
    fraccion_singular: str
    fraccion_plural: str


SOLES = Moneda("sol", "soles", "céntimo", "céntimos")
DOLARES = Moneda("dólar", "dólares", "centavo", "centavos")
_CIFRA = r"\d(?:[\d.,]*\d)?"
_PATRON_SOLES = re.compile(rf"(?<![\w/])S/\.?\s*({_CIFRA})", re.IGNORECASE)
_PATRON_DOLARES = re.compile(rf"(?<!\w)(?:US\s?\$|USD\s?|\$)\s*({_CIFRA})", re.IGNORECASE)
_PATRON_MILES_CON_COMA = re.compile(r"(?<![\d.,])\d{1,3}(?:,\d{3})+(?:\.\d+)?(?![\d,])")


def parsear_monto(cifra: str) -> tuple[int, int] | None:
    """«1,500.50», «1.500,50», «1500» o «12.5» → (entero, céntimos); None si no se entiende.

    Si solo hay un tipo de separador, aparece una vez y no lo siguen exactamente 3 cifras, es decimal;
    en cualquier otro caso es separador de miles («1,500» y «1.500» valen mil quinientos).
    """
    punto, coma = cifra.rfind("."), cifra.rfind(",")
    decimal = ""
    if punto >= 0 and coma >= 0:
        decimal = "." if punto > coma else ","
    elif punto >= 0 or coma >= 0:
        separador = "." if punto >= 0 else ","
        if cifra.count(separador) == 1 and len(cifra.rsplit(separador, 1)[1]) != 3:
            decimal = separador
    entero, _, fraccion = cifra.rpartition(decimal) if decimal else (cifra, "", "")
    entero = entero.replace(".", "").replace(",", "")
    if not entero.isdecimal() or (fraccion and not fraccion.isdecimal()):
        return None
    return int(entero), int((fraccion + "00")[:2]) if fraccion else 0


def monto_en_palabras(entero: int, centimos: int, moneda: Moneda) -> str:
    """(1500, 50, SOLES) → «mil quinientos soles con cincuenta céntimos»; (1, 0, SOLES) → «un sol»."""
    partes: list[str] = []
    if entero or not centimos:
        de = " de" if entero and entero % 1_000_000 == 0 else ""  # «un millón de soles»
        nombre = moneda.singular if entero == 1 else moneda.plural
        partes.append(f"{numero_en_palabras(entero, apocope=True)}{de} {nombre}")
    if centimos:
        nombre = moneda.fraccion_singular if centimos == 1 else moneda.fraccion_plural
        partes.append(f"{numero_en_palabras(centimos, apocope=True)} {nombre}")
    return " con ".join(partes)


def _leer_monto(coincidencia: re.Match[str], moneda: Moneda) -> str:
    monto = parsear_monto(coincidencia.group(1))
    if monto is None:
        return coincidencia.group(0)
    try:
        return monto_en_palabras(*monto, moneda)
    except ValueError:
        return coincidencia.group(0)


def normalizar_cifras(texto: str) -> tuple[str, int]:
    """Montos en soles y dólares → palabras; «1,500» → «1500» para que XTTS lo lea como mil quinientos."""
    total = 0
    for patron, moneda in ((_PATRON_SOLES, SOLES), (_PATRON_DOLARES, DOLARES)):
        texto, cambios = patron.subn(lambda m, moneda=moneda: _leer_monto(m, moneda), texto)
        total += cambios
    texto, cambios = _PATRON_MILES_CON_COMA.subn(lambda m: m.group(0).replace(",", ""), texto)
    return texto, total + cambios


# ═══ Pronunciación ══════════════════════════════════════════════════════════════

LARGO_MAXIMO_REGLA = 100


@dataclass(frozen=True)
class Regla:
    """«texto = lectura»: cada aparición de `texto` (palabra completa, sin distinguir mayúsculas) se lee como `lectura`."""

    texto: str
    lectura: str


def leer_reglas(fuente: str) -> tuple[list[Regla], list[str]]:
    """Reglas separadas por «;» o por líneas; una línea que empieza con «#» es un comentario.

    Si un texto se repite, gana la última regla: así el formulario puede corregir lo que dice el archivo.
    """
    reglas: dict[str, Regla] = {}
    avisos: list[str] = []
    for linea in fuente.splitlines():
        if linea.lstrip().startswith("#"):
            continue
        for crudo in linea.split(";"):
            if not crudo.strip():
                continue
            texto, igual, lectura = (" ".join(parte.split()) for parte in crudo.partition("="))
            if not igual or not texto:
                avisos.append(f"Regla ignorada (formato «texto = lectura»): «{crudo.strip()}»")
                continue
            if len(texto) > LARGO_MAXIMO_REGLA:
                avisos.append(f"Regla ignorada (más de {LARGO_MAXIMO_REGLA} caracteres): «{texto[:40]}…»")
                continue
            texto = unicodedata.normalize("NFC", texto)
            reglas.pop(texto.lower(), None)
            reglas[texto.lower()] = Regla(texto, unicodedata.normalize("NFC", lectura))
    return list(reglas.values()), avisos


class _NodoTrie:
    __slots__ = ("hijos", "fin")

    def __init__(self) -> None:
        self.hijos: dict[str, _NodoTrie] = {}
        self.fin = False


def _patron_de_trie(nodo: _NodoTrie) -> str:
    """Regex equivalente a la trie: prefijos comunes factorizados; el «?» codicioso prueba primero la clave más larga."""
    ramas = [re.escape(caracter) + _patron_de_trie(hijo) for caracter, hijo in sorted(nodo.hijos.items())]
    if not ramas:
        return ""
    cuerpo = ramas[0] if len(ramas) == 1 else "(?:" + "|".join(ramas) + ")"
    return f"(?:{cuerpo})?" if nodo.fin else cuerpo


class Pronunciador:
    """Aplica todas las reglas en una sola pasada; si dos empiezan igual («Machu» / «Machu Picchu»), gana la más larga.

    Los textos se guardan en una trie que se compila a una única regex con los prefijos comunes factorizados:
    el trabajo por posición depende del largo de las claves, no de cuántas reglas haya.
    """

    def __init__(self, reglas: Sequence[Regla]) -> None:
        self.cantidad = len(reglas)
        self._lecturas = {regla.texto.lower(): regla.lectura for regla in reglas}
        raiz = _NodoTrie()
        for clave in self._lecturas:
            nodo = raiz
            for caracter in clave:
                nodo = nodo.hijos.setdefault(caracter, _NodoTrie())
            nodo.fin = True
        self._patron = (
            re.compile(rf"(?<!\w)(?:{_patron_de_trie(raiz)})(?!\w)", re.IGNORECASE) if self._lecturas else None
        )

    def aplicar(self, texto: str) -> tuple[str, int]:
        """Texto con las lecturas aplicadas y cuántos reemplazos hubo."""
        texto = re.sub(r"[ \t]+", " ", unicodedata.normalize("NFC", texto))
        if self._patron is None:
            return texto, 0
        return self._patron.subn(lambda m: self._lecturas.get(m.group(0).lower(), m.group(0)), texto)


# ─── Ejecución ───
print("✅ Funciones de texto cargadas.")

In [ ]:
# @title 3 · Cargar el modelo XTTS-v2 en la GPU { display-mode: "form" }
# @markdown **Uso asumido por defecto: no comercial** (personal, educativo o de investigación). Es el único que permite la licencia de XTTS-v2, la Coqui Public Model License ([CPML](https://tts-hub.github.io/cpml)); con la casilla marcada, el cuaderno la acepta en tu nombre.
uso_no_comercial = True  # @param {type:"boolean"}
# @markdown *Desmárcala solo si vas a darle a los audios un uso comercial (anuncios pagados, productos o servicios que vendes): el cuaderno se detendrá antes de cargar el modelo. El modelo (~1,9 GB) se descarga una vez por sesión.*

"""Celda 3 · Motor de voz: adaptador sobre XTTS-v2. Todo lo específico del modelo vive en esta celda."""

import os
import time
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

if not {"montar_drive", "leer_wav", "dividir_texto"} <= globals().keys():  # una función de cada celda 2a, 2b y 2c
    raise RuntimeError("Ejecuta primero las celdas 1, 2a, 2b y 2c.")

NOMBRE_MODELO = "tts_models/multilingual/multi-dataset/xtts_v2"
SEGUNDOS_ESTILO = 30  # audio de referencia que analiza el GPT para imitar entonación y ritmo
TRAMO_ESTILO = 4  # se analiza en tramos de 4 s que luego se promedian: más estable


@dataclass(frozen=True)
class PerfilVoz:
    """Huella de una voz: latentes de estilo (GPT) y embedding del hablante (decodificador)."""

    latente_estilo: Any
    embedding: Any
    archivos: tuple[str, ...]
    segundos: float


@dataclass(frozen=True)
class AjustesSintesis:
    """Parámetros de una generación."""

    idioma: str
    velocidad: float = 1.0
    temperatura: float = 0.75
    top_p: float = 0.85
    top_k: int = 50
    penalizacion_repeticion: float = 5.0


class MotorXTTS:
    """Crea perfiles de voz y sintetiza frases con un XTTS-v2 ya cargado en la GPU."""

    def __init__(self, tts: Any) -> None:
        import torch

        self._torch = torch
        self.modelo = tts.synthesizer.tts_model
        self.frecuencia = int(getattr(self.modelo.args, "output_sample_rate", 24000))
        self.idiomas = frozenset(self.modelo.config.languages)

    def limite_caracteres(self, idioma: str) -> int:
        """Máximo de caracteres por fragmento que XTTS lee sin cortar el audio."""
        limites = getattr(self.modelo.tokenizer, "char_limits", {})
        return int(limites.get(idioma.split("-")[0], 200))

    def crear_voz(self, wavs: Sequence[Path], segundos_por_archivo: float) -> PerfilVoz:
        """Equivale a Xtts.get_conditioning_latents, pero lee los WAV con NumPy (sin torchaudio.load)."""
        torch, dispositivo = self._torch, self.modelo.device
        audios, embeddings, total = [], [], 0.0
        with torch.inference_mode():
            for ruta in wavs:
                muestras, frecuencia = leer_wav(ruta)
                if frecuencia != FRECUENCIA_REFERENCIA:
                    raise ValueError(f"{ruta.name}: {frecuencia} Hz (se esperaban {FRECUENCIA_REFERENCIA})")
                muestras = muestras[: int(frecuencia * segundos_por_archivo)]
                audio = torch.from_numpy(np.ascontiguousarray(muestras)).unsqueeze(0).to(dispositivo)
                embeddings.append(self.modelo.get_speaker_embedding(audio, frecuencia))
                audios.append(audio)
                total += muestras.size / frecuencia
            estilo = self.modelo.get_gpt_cond_latents(
                torch.cat(audios, dim=-1), FRECUENCIA_REFERENCIA, length=SEGUNDOS_ESTILO, chunk_length=TRAMO_ESTILO
            )
            embedding = torch.stack(embeddings).mean(dim=0)
        return PerfilVoz(estilo, embedding, tuple(str(r) for r in wavs), total)

    def fijar_semilla(self, semilla: int) -> None:
        """Hace repetible el muestreo aleatorio del modelo."""
        self._torch.manual_seed(semilla)
        if self._torch.cuda.is_available():
            self._torch.cuda.manual_seed_all(semilla)

    def sintetizar(self, texto: str, voz: PerfilVoz, ajustes: AjustesSintesis) -> np.ndarray:
        """Un fragmento de texto → forma de onda float32 a `self.frecuencia` Hz.

        transformers exige float en temperatura y penalización, y Colab escribe «7» (no «7.0») cuando
        un deslizador cae en un entero: por eso se convierte cada valor a su tipo aquí.
        """
        salida = self.modelo.inference(
            texto,
            ajustes.idioma,
            voz.latente_estilo,
            voz.embedding,
            temperature=float(ajustes.temperatura),
            top_p=float(ajustes.top_p),
            top_k=int(ajustes.top_k),
            repetition_penalty=float(ajustes.penalizacion_repeticion),
            length_penalty=1.0,
            speed=float(ajustes.velocidad),
            enable_text_splitting=False,  # el texto ya llega dividido por dividir_texto()
        )
        return np.asarray(salida["wav"], dtype=np.float32).reshape(-1)


def confirmar_licencia(no_comercial: bool) -> None:
    """Aplica el caso de uso declarado en la casilla y lo deja a la vista en cada ejecución (transparencia)."""
    if not no_comercial:
        os.environ.pop("COQUI_TOS_AGREED", None)
        raise ErrorUsuario(
            "Desmarcaste «uso_no_comercial»: la licencia de XTTS-v2 (CPML) no permite el uso comercial, así que "
            "el modelo no se carga. Si fue sin querer, vuelve a marcar la casilla y ejecuta esta celda otra vez."
        )
    os.environ["COQUI_TOS_AGREED"] = "1"  # forma prevista por coqui-tts para aceptar la CPML sin preguntas interactivas
    print("ℹ️ Licencia: CPML aceptada para uso no comercial (casilla «uso_no_comercial» de esta celda).")


def cargar_xtts() -> Any:
    """Descarga (si hace falta) XTTS-v2 y lo sube a la GPU. Es lo único lento de esta celda."""
    import torch

    if not torch.cuda.is_available():
        raise ErrorUsuario("PyTorch no detecta la GPU. Cambia el entorno a «T4 GPU» y ejecuta desde la celda 1.")
    from TTS.api import TTS

    print("⏳ Cargando XTTS-v2 en la GPU (la primera vez descarga ~1,9 GB)…")
    inicio = time.perf_counter()
    tts = TTS(NOMBRE_MODELO, progress_bar=True).to("cuda")
    print(f"✅ Modelo listo en {torch.cuda.get_device_name(0)} ({time.perf_counter() - inicio:.0f} s)")
    return tts


# ─── Ejecución ───
MOTOR = None  # sin la licencia aceptada, las celdas 4 y 5 no tienen motor que usar
confirmar_licencia(uso_no_comercial)
if globals().get("_XTTS") is None:
    _XTTS = cargar_xtts()
else:
    print("✅ XTTS-v2 ya estaba cargado en esta sesión.")
MOTOR = MotorXTTS(_XTTS)

In [ ]:
# @title 4 · Tu voz de referencia (Drive o tu computadora) { display-mode: "form" }
origen_del_audio = "Google Drive"  # @param ["Google Drive", "Subir desde mi computadora"]
# @markdown *Si Drive no se conecta, elige «Subir desde mi computadora»: al ejecutar la celda aparecerá un botón para elegir el archivo.*
# @markdown Con Drive, escribe el **nombre del audio** tal como está en tu Drive (`mi_voz.m4a`), una **ruta** dentro de *Mi unidad* (`Audios/mi_voz.m4a`) o una **carpeta** con varios audios tuyos (`Audios/Mi voz`).
audio_en_drive = "mi_voz.wav"  # @param {type:"string"}
# @markdown ---
# @markdown **Preparación del audio**
limpiar_audio = True  # @param {type:"boolean"}
# @markdown *Recorta los silencios del inicio y del final, quita retumbes graves y nivela el volumen.*
reducir_ruido = False  # @param {type:"boolean"}
# @markdown *Actívalo solo si se oye un ruido de fondo constante (ventilador, aire acondicionado).*
segundos_por_archivo = 60  # @param {type:"slider", min:6, max:120, step:2}
# @markdown *Segundos que se analizan de cada audio; entre 30 y 60 suele dar el mejor parecido.*

"""Celda 4 · Voz de referencia: Drive o tu computadora → FFmpeg (WAV limpio) → perfil de voz en la GPU."""

from collections.abc import Sequence
from pathlib import Path

if globals().get("MOTOR") is None:
    raise RuntimeError("Ejecuta primero las celdas 1 a 3.")

SEGUNDOS_MINIMOS = 3.0
SEGUNDOS_RECOMENDADOS = 15.0
MAXIMO_ARCHIVOS = 12
SUBIR = "Subir desde mi computadora"


def audios_desde_drive(entrada: str) -> list[Path]:
    """El audio indicado en Drive, o los audios de la carpeta indicada."""
    montar_drive()
    return listar_medios(resolver_en_drive(entrada, extensiones=EXTENSIONES_MEDIOS, incluir_carpetas=True), MAXIMO_ARCHIVOS)


def audios_desde_computadora() -> list[Path]:
    """Muestra el botón de Colab para elegir uno o varios audios o videos de tu computadora (sin pasar por Drive)."""
    from google.colab import files

    carpeta = preparar_carpeta(CARPETA_TRABAJO / "subidas", vaciar=True)
    print("⬆️ Elige tu audio con el botón de abajo (puedes elegir varios)…")
    subidos = sorted(Path(nombre) for nombre in files.upload(target_dir=str(carpeta)))
    audios = [ruta for ruta in subidos if ruta.suffix.casefold() in EXTENSIONES_MEDIOS]
    if not audios:
        raise ErrorUsuario("No subiste ningún audio o video (WAV, MP3, M4A, OPUS, MP4…). Ejecuta la celda otra vez.")
    if len(audios) > MAXIMO_ARCHIVOS:
        print(f"ℹ️ Subiste {len(audios)} archivos; uso los primeros {MAXIMO_ARCHIVOS}.")
    return audios[:MAXIMO_ARCHIVOS]


def preparar_referencias(
    origenes: Sequence[Path], carpeta: Path, *, filtros: str, max_segundos: float
) -> list[tuple[Path, float]]:
    """Convierte cada audio o video en un WAV listo para el modelo; devuelve (ruta, segundos útiles)."""
    preparadas = []
    for indice, origen in enumerate(origenes, start=1):
        destino = carpeta / f"referencia_{indice:02d}.wav"
        preparar_referencia(origen, destino, filtros=filtros, max_segundos=max_segundos)
        muestras, frecuencia = leer_wav(destino)
        segundos = muestras.size / frecuencia
        if segundos < 0.5:
            print(f"   ⚠️ {origen.name}: casi no tiene voz; lo omito.")
            continue
        print(f"   🎧 {origen.name} → {segundos:.1f} s útiles")
        preparadas.append((destino, segundos))
    return preparadas


def crear_perfil(archivos: Sequence[Path], *, limpiar: bool, reducir: bool, segundos: float) -> PerfilVoz:
    """Prepara los audios y calcula la huella de la voz."""
    carpeta = preparar_carpeta(CARPETA_TRABAJO / "referencias", vaciar=True)
    print(f"🎚️ Preparando {len(archivos)} archivo(s)…")
    preparadas = preparar_referencias(
        archivos, carpeta, filtros=filtros_referencia(limpiar, reducir), max_segundos=segundos
    )
    total = sum(s for _, s in preparadas)
    if total < SEGUNDOS_MINIMOS:
        raise ErrorUsuario(f"Solo hay {total:.1f} s de voz útil. Usa una grabación de al menos 10 s (ideal 30–60 s).")
    if total < SEGUNDOS_RECOMENDADOS:
        print(f"⚠️ {total:.1f} s es poco: el parecido mejora mucho con 30–60 s de grabación.")
    voz = MOTOR.crear_voz([ruta for ruta, _ in preparadas], segundos)
    print(f"✅ Voz lista: {len(preparadas)} archivo(s), {formatear_duracion(total)} analizados. Así suena la referencia:")
    mostrar_reproductor(preparadas[0][0])
    return voz


# ─── Ejecución ───
VOZ = crear_perfil(
    audios_desde_computadora() if origen_del_audio == SUBIR else audios_desde_drive(audio_en_drive),
    limpiar=limpiar_audio,
    reducir=reducir_ruido,
    segundos=segundos_por_archivo,
)

In [ ]:
# @title 5 · Generar audio con tu voz { display-mode: "form" }
texto = "Hola, esta es una prueba de mi voz clonada. Puedo leer cualquier texto que escriba aquí, con pausas naturales entre frases."  # @param {type:"string"}
# @markdown *Silencios: `[pausa]` (1 s) o `[pausa 2.5]`. Otro idioma con tu misma voz: `[en]Where is the library?[/en]` (también `[pt]`, `[fr]`, `[it]`, `[de]`…). Para textos largos usa un archivo:*
archivo_de_texto = ""  # @param {type:"string"}
# @markdown *Opcional: nombre de un `.txt` en tu Drive. Si lo llenas, se lee ese archivo en lugar del cuadro de arriba (cada salto de línea es un párrafo).*
idioma = "Español"  # @param ["Español", "English", "Português", "Français", "Italiano", "Deutsch", "Nederlands", "Polski", "Türkçe", "Русский", "Čeština", "Magyar", "العربية", "हिन्दी"]
# @markdown ---
# @markdown **Estilo**
velocidad = 1.0  # @param {type:"slider", min:0.6, max:1.6, step:0.05}
temperatura = 0.75  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
# @markdown *Más baja: más estable y monótona. Más alta: más expresiva, pero con más riesgo de errores.*
pausa_entre_frases = 0.3  # @param {type:"slider", min:0.0, max:1.5, step:0.05}
# @markdown ---
# @markdown **Tomas**
tomas = 1  # @param {type:"slider", min:1, max:4, step:1}
# @markdown *Versiones distintas del mismo texto para quedarte con la mejor en la celda 6. Cada toma tarda lo mismo que una generación.*
semilla = -1  # @param {type:"integer"}
# @markdown *`-1` = al azar. Cada toma imprime su semilla: cópiala aquí (con `tomas` en 1) para repetir ese resultado.*
# @markdown ---
# @markdown **Pronunciación**
leer_montos_y_cifras = True  # @param {type:"boolean"}
# @markdown *Solo en español: «S/ 1,500.50» → «mil quinientos soles con cincuenta céntimos», «US$ 21» → «veintiún dólares» y «1,500» → «mil quinientos» (sin esto, el modelo lee «uno punto cinco»).*
pronunciacion = ""  # @param {type:"string"}
# @markdown *Reglas `texto = cómo se lee`, separadas por `;`. Ej.: `DNI = de ene i; Jr. = jirón; Av. = avenida`.*
archivo_de_pronunciacion = ""  # @param {type:"string"}
# @markdown *Para listas largas: un `.txt` en tu Drive con una regla por línea (las que empiezan con `#` son comentarios). Las reglas del cuadro de arriba tienen prioridad.*
# @markdown ---
# @markdown **Salida**
nombre_salida = "mi_voz_clonada"  # @param {type:"string"}
formato = "mp3"  # @param ["mp3", "wav"]
volumen = "Parejo: -16 LUFS (voz)"  # @param ["Parejo: -16 LUFS (voz)", "Alto: -14 LUFS (redes)", "Suave: -20 LUFS", "Sin nivelar"]
# @markdown *Todas las tomas y generaciones salen con el mismo volumen percibido, sin que los picos pasen de −1 dBFS.*
guardar_en_drive = True  # @param {type:"boolean"}
carpeta_en_drive = "Clonador de voz"  # @param {type:"string"}
descargar_al_terminar = False  # @param {type:"boolean"}
# @markdown *Con varias tomas, el guardado en Drive y la descarga se hacen desde la celda 6, con la toma elegida.*
# @markdown ---
# @markdown **Avanzado** *(muestreo del modelo)*
top_p = 0.85  # @param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 50  # @param {type:"slider", min:1, max:100, step:1}
penalizacion_repeticion = 5.0  # @param {type:"slider", min:1.0, max:15.0, step:0.5}

"""Celda 5 · Generación: texto → idiomas y pronunciación → fragmentos → XTTS-v2 → volumen → una o varias tomas."""

import random
import shutil
import time
from collections import Counter
from collections.abc import Sequence
from dataclasses import dataclass, replace
from pathlib import Path

import numpy as np

if globals().get("VOZ") is None or globals().get("MOTOR") is None:
    raise RuntimeError("Ejecuta primero las celdas 1 a 4.")

IDIOMAS = {
    "Español": "es", "English": "en", "Português": "pt", "Français": "fr", "Italiano": "it",
    "Deutsch": "de", "Nederlands": "nl", "Polski": "pl", "Türkçe": "tr", "Русский": "ru",
    "Čeština": "cs", "Magyar": "hu", "العربية": "ar", "हिन्दी": "hi",
}  # XTTS también habla chino, japonés y coreano, pero requieren: pip install "coqui-tts[zh,ja,ko]"
IDIOMAS_DISPONIBLES = frozenset(IDIOMAS.values())
VOLUMENES = {"Parejo: -16 LUFS (voz)": -16.0, "Alto: -14 LUFS (redes)": -14.0, "Suave: -20 LUFS": -20.0, "Sin nivelar": None}
MARGEN_LIMITE = 10  # caracteres por debajo del límite del modelo, por seguridad
REINTENTOS_POR_MITADES = 2  # un fragmento que el modelo rechaza se parte en mitades hasta 2 veces
SEMILLA_MAXIMA = 2**31 - 1


@dataclass(frozen=True)
class OpcionesTexto:
    pausa_frase: float
    cifras: bool
    pronunciador: Pronunciador | None


@dataclass(frozen=True)
class OpcionesSalida:
    nombre: str
    formato: str
    sonoridad: float | None  # LUFS objetivo; None = solo normalizar el pico
    guardar_en_drive: bool
    carpeta_en_drive: str
    descargar: bool


@dataclass
class Toma:
    """Un resultado completo; `en_drive` se llena al copiarlo a Drive (en la celda 5 o en la 6)."""

    numero: int
    semilla: int
    archivo: Path
    en_drive: Path | None = None


@dataclass(frozen=True)
class Generacion:
    tomas: list[Toma]
    salida: OpcionesSalida


# ── Texto ──


def leer_texto(texto_formulario: str, archivo: str) -> str:
    """El texto del formulario o, si se indicó, el contenido de un .txt de Drive."""
    if not archivo.strip():
        return texto_formulario
    montar_drive()
    ruta = resolver_en_drive(archivo, extensiones=EXTENSIONES_TEXTO, incluir_carpetas=False)
    contenido = leer_texto_de_archivo(ruta)
    print(f"📄 Texto de {describir_ruta(ruta)}: {len(contenido)} caracteres.")
    return contenido


def cargar_pronunciador(reglas_formulario: str, archivo: str) -> Pronunciador | None:
    """Reglas del .txt de Drive (si se indicó) y después las del formulario, que tienen prioridad."""
    fuente = ""
    if archivo.strip():
        montar_drive()
        fuente = leer_texto_de_archivo(resolver_en_drive(archivo, extensiones=EXTENSIONES_TEXTO, incluir_carpetas=False))
    reglas, avisos = leer_reglas(f"{fuente}\n{reglas_formulario}")
    for aviso in avisos:
        print(f"⚠️ {aviso}")
    return Pronunciador(reglas) if reglas else None


def preparar_segmentos(texto: str, principal: str, opciones: OpcionesTexto) -> list[Segmento]:
    """Separa el texto en tramos por idioma; en cada uno adapta montos y cifras (si es español) y aplica las reglas
    de pronunciación (si es el idioma principal), y lo divide con el límite de caracteres de su idioma."""
    tramos, avisos = dividir_por_idioma(texto, principal, IDIOMAS_DISPONIBLES)
    for aviso in avisos:
        print(f"⚠️ {aviso}")
    segmentos: list[Segmento] = []
    cifras = reemplazos = 0
    anterior = ""
    for tramo in tramos:
        contenido = tramo.texto
        if opciones.cifras and tramo.idioma == "es":
            contenido, cambios = normalizar_cifras(contenido)
            cifras += cambios
        if opciones.pronunciador is not None and tramo.idioma == principal:
            contenido, cambios = opciones.pronunciador.aplicar(contenido)
            reemplazos += cambios
        nuevos = dividir_texto(contenido, MOTOR.limite_caracteres(tramo.idioma) - MARGEN_LIMITE, opciones.pausa_frase)
        if segmentos:  # frontera entre idiomas: pausa de frase o de párrafo, más los [pausa] que haya al inicio
            frontera = opciones.pausa_frase
            if hay_salto_de_parrafo(anterior, tramo.texto):
                frontera = pausa_de_parrafo(opciones.pausa_frase)
            pausa = max(segmentos[-1].pausa, frontera) if nuevos else segmentos[-1].pausa
            segmentos[-1] = replace(segmentos[-1], pausa=pausa + pausa_inicial(contenido))
        segmentos.extend(replace(segmento, idioma=tramo.idioma) for segmento in nuevos)
        anterior = tramo.texto if nuevos else anterior + tramo.texto
    if not segmentos:
        raise ErrorUsuario("No hay texto que leer: escribe algo en «texto» o indica un archivo .txt.")
    if cifras:
        print(f"💲 Montos y cifras: {cifras} adaptado(s) para que se lean bien.")
    if opciones.pronunciador is not None:
        print(f"🔤 Pronunciación: {opciones.pronunciador.cantidad} regla(s) · {reemplazos} reemplazo(s) en el texto.")
    por_idioma = Counter(segmento.idioma for segmento in segmentos)
    if len(por_idioma) > 1:
        print("🌐 Idiomas: " + " · ".join(f"{codigo} ({n} fragmento(s))" for codigo, n in por_idioma.items()))
    return segmentos


# ── Síntesis ──


def sintetizar_tolerante(texto: str, ajustes: AjustesSintesis, profundidad: int = 0) -> np.ndarray | None:
    """Sintetiza un fragmento; si el modelo lo rechaza, reintenta por mitades en vez de perder todo el audio.

    Ejemplo típico: XTTS admite menos de 400 tokens y los números se expanden a palabras («1999» → «mil
    novecientos noventa y nueve») después de dividir el texto por caracteres.
    """
    try:
        return MOTOR.sintetizar(texto, VOZ, ajustes)
    except Exception as error:  # noqa: BLE001 — se informa y se continúa con el resto del texto
        mitades = partir_por_la_mitad(texto) if profundidad < REINTENTOS_POR_MITADES else None
        if mitades is None:
            print(f"\n⚠️ Se omitió «{texto[:60]}» ({type(error).__name__}: {error})")
            return None
        partes = [sintetizar_tolerante(mitad, ajustes, profundidad + 1) for mitad in mitades]
        partes = [parte for parte in partes if parte is not None and parte.size]
        return np.concatenate(partes) if partes else None


def sintetizar_segmentos(
    segmentos: Sequence[Segmento], ajustes: AjustesSintesis
) -> tuple[list[np.ndarray], list[float]]:
    """Genera cada fragmento, le quita los silencios de los bordes y anota la pausa que va después."""
    fragmentos, pausas = [], []
    for numero, segmento in enumerate(segmentos, start=1):
        print(f"\r🗣️ Fragmento {numero}/{len(segmentos)}…", end="", flush=True)
        if segmento.idioma not in ("", ajustes.idioma):
            onda = sintetizar_tolerante(segmento.texto, replace(ajustes, idioma=segmento.idioma))
        else:
            onda = sintetizar_tolerante(segmento.texto, ajustes)
        onda = recortar_silencios(onda, MOTOR.frecuencia) if onda is not None else None
        if onda is None or onda.size == 0:
            print(f"\n⚠️ El fragmento {numero} quedó sin audio: «{segmento.texto[:60]}»")
            continue
        fragmentos.append(aplicar_fundidos(onda, MOTOR.frecuencia))
        pausas.append(segmento.pausa)
    print("\r", end="")
    return fragmentos, pausas


def semillas_para(semilla: int, cantidad: int) -> list[int]:
    """La semilla indicada (o una al azar) y las consecutivas: cada toma se puede repetir por separado."""
    base = semilla if semilla >= 0 else random.randrange(1, SEMILLA_MAXIMA)
    return [(base + desplazamiento) % SEMILLA_MAXIMA for desplazamiento in range(cantidad)]


def generar_toma(
    numero: int, total: int, segmentos: Sequence[Segmento], ajustes: AjustesSintesis, semilla: int, salida: OpcionesSalida
) -> Toma:
    """Sintetiza el texto completo con una semilla y lo exporta como archivo en Colab."""
    MOTOR.fijar_semilla(semilla)
    inicio = time.perf_counter()
    fragmentos, pausas = sintetizar_segmentos(segmentos, ajustes)
    if not fragmentos:
        raise ErrorUsuario("El modelo no produjo audio. Prueba con otra semilla o revisa el texto.")
    audio = concatenar_con_pausas(fragmentos, pausas, MOTOR.frecuencia)
    if salida.sonoridad is None:
        audio, volumen = normalizar_pico(audio), ""
    else:
        audio = nivelar_volumen(audio, MOTOR.frecuencia, salida.sonoridad)
        volumen = f" · {sonoridad_integrada(audio, MOTOR.frecuencia):.1f} LUFS"
    base = nombre_seguro(salida.nombre)
    archivo = exportar(audio, f"{base}_toma{numero}" if total > 1 else base, salida.formato)
    prefijo = f"Toma {numero}/{total} · " if total > 1 else ""
    print(
        f"✅ {prefijo}{len(fragmentos)} fragmento(s) · {formatear_duracion(audio.size / MOTOR.frecuencia)} de audio"
        f"{volumen} · generado en {time.perf_counter() - inicio:.1f} s · semilla {semilla}"
    )
    if len(fragmentos) < len(segmentos):
        print(f"⚠️ {len(segmentos) - len(fragmentos)} fragmento(s) quedaron sin audio (ver avisos): revisa ese texto o cambia la semilla.")
    return Toma(numero, semilla, archivo)


# ── Salida ──


def exportar(audio: np.ndarray, nombre: str, formato: str) -> Path:
    """Escribe el audio en /content/clonador_voz/salidas como WAV o MP3, sin sobrescribir nada."""
    carpeta = preparar_carpeta(CARPETA_TRABAJO / "salidas")
    destino = ruta_libre(carpeta, nombre_seguro(nombre), f".{formato}")
    if formato == "wav":
        escribir_wav(destino, audio, MOTOR.frecuencia)
        return destino
    temporal = carpeta / "_temporal.wav"
    escribir_wav(temporal, audio, MOTOR.frecuencia)
    try:
        convertir_a_mp3(temporal, destino)
    finally:
        temporal.unlink(missing_ok=True)
    return destino


def copiar_a_drive(archivo: Path, carpeta_texto: str, nombre: str) -> Path:
    """Copia el archivo a la carpeta indicada de Drive (la crea si no existe) con el nombre dado, sin sobrescribir."""
    montar_drive()
    carpeta = preparar_carpeta(ruta_en_mi_unidad(carpeta_texto.strip() or "Clonador de voz"))
    destino = ruta_libre(carpeta, nombre_seguro(nombre), archivo.suffix)
    shutil.copyfile(archivo, destino)
    return destino


def guardar_toma_en_drive(toma: Toma, salida: OpcionesSalida) -> Path:
    """Copia la toma a Drive con el nombre de salida (sin «_tomaN»); si ya se copió, no la duplica."""
    if toma.en_drive is None:
        toma.en_drive = copiar_a_drive(toma.archivo, salida.carpeta_en_drive, salida.nombre)
        print(f"💾 Guardado en {describir_ruta(toma.en_drive)}")
    else:
        print(f"ℹ️ Ya estaba guardada en {describir_ruta(toma.en_drive)}")
    return toma.en_drive


def intentar_guardar_en_drive(toma: Toma, salida: OpcionesSalida) -> None:
    """Como guardar_toma_en_drive, pero si Drive no se conecta el audio ya generado no se pierde: queda en Colab."""
    try:
        guardar_toma_en_drive(toma, salida)
    except DriveNoDisponible as error:
        print(
            f"⚠️ No se pudo guardar en Drive ({error.detalle}). El audio quedó en Colab: reintenta el guardado con la "
            "celda 6 (elige «1»; revisa antes la cuenta y los permisos de Drive) o márcala con «descargar_tambien»."
        )


def descargar(archivo: Path) -> None:
    from google.colab import files

    files.download(str(archivo))


# ── Orquestación ──


def generar_audio(
    texto_final: str, ajustes: AjustesSintesis, opciones: OpcionesTexto, *, semilla: int, tomas: int, salida: OpcionesSalida
) -> Generacion:
    """Genera una o varias tomas. Con una sola, la entrega de inmediato; con varias, la elección queda para la celda 6."""
    if ajustes.idioma not in MOTOR.idiomas:
        raise ErrorUsuario(f"El modelo no admite el idioma «{ajustes.idioma}».")
    segmentos = preparar_segmentos(texto_final, ajustes.idioma, opciones)
    total = max(1, tomas)
    generacion = Generacion([], salida)
    for numero, semilla_toma in enumerate(semillas_para(semilla, total), start=1):
        generacion.tomas.append(generar_toma(numero, total, segmentos, ajustes, semilla_toma, salida))
        if total > 1:
            mostrar_reproductor(generacion.tomas[-1].archivo)
    if total > 1:
        print(f"👉 Escucha las {total} tomas y guarda la mejor con la celda 6.")
        return generacion
    toma = generacion.tomas[0]
    print(f"   Archivo en Colab: {toma.archivo}")
    if salida.guardar_en_drive:
        intentar_guardar_en_drive(toma, salida)
    mostrar_reproductor(toma.archivo)
    if salida.descargar:
        descargar(toma.archivo)
    return generacion


# ─── Ejecución ───
ULTIMA_GENERACION = generar_audio(
    leer_texto(texto, archivo_de_texto),
    AjustesSintesis(
        idioma=IDIOMAS[idioma],
        velocidad=velocidad,
        temperatura=temperatura,
        top_p=top_p,
        top_k=int(top_k),
        penalizacion_repeticion=penalizacion_repeticion,
    ),
    OpcionesTexto(
        pausa_frase=pausa_entre_frases,
        cifras=leer_montos_y_cifras,
        pronunciador=cargar_pronunciador(pronunciacion, archivo_de_pronunciacion),
    ),
    semilla=int(semilla),
    tomas=int(tomas),
    salida=OpcionesSalida(
        nombre=nombre_salida,
        formato=formato,
        sonoridad=VOLUMENES[volumen],
        guardar_en_drive=guardar_en_drive,
        carpeta_en_drive=carpeta_en_drive,
        descargar=descargar_al_terminar,
    ),
)

In [ ]:
# @title 6 · Guardar la toma elegida { display-mode: "form" }
# @markdown Si en la celda 5 pediste varias **tomas**, elige aquí la mejor: se guarda en tu Drive con la carpeta y el nombre de salida de la celda 5.
toma_elegida = "—"  # @param ["—", "1", "2", "3", "4"]
descargar_tambien = False  # @param {type:"boolean"}

"""Celda 6 · Elección: guarda en Drive (y descarga, si se pide) una toma de la última generación."""

if globals().get("ULTIMA_GENERACION") is None:
    raise RuntimeError("Ejecuta primero la celda 5.")


def guardar_toma_elegida(eleccion: str, generacion: Generacion, *, descargar_archivo: bool) -> Toma | None:
    """Guarda la toma elegida. Con «—» solo orienta, para que «Ejecutar todas» no guarde nada por su cuenta."""
    cantidad = len(generacion.tomas)
    if not eleccion.isdecimal():
        unica = generacion.tomas[0]
        if cantidad > 1:
            print(f"👉 Escucha las {cantidad} tomas de la celda 5, elige una aquí y vuelve a ejecutar esta celda.")
        elif unica.en_drive is not None:
            print(f"ℹ️ La celda 5 generó una sola toma y ya está en {describir_ruta(unica.en_drive)}.")
        else:
            print("ℹ️ La celda 5 generó una sola toma; elige «1» si quieres guardarla en Drive.")
        return None
    numero = int(eleccion)
    if not 1 <= numero <= cantidad:
        raise ErrorUsuario(f"La última generación tiene {cantidad} toma(s): elige un número entre 1 y {cantidad}.")
    toma = generacion.tomas[numero - 1]
    print(f"🎯 Toma {numero} · semilla {toma.semilla}")
    try:
        guardar_toma_en_drive(toma, generacion.salida)
    finally:  # aunque Drive falle, la descarga pedida se hace
        if descargar_archivo:
            descargar(toma.archivo)
    return toma


# ─── Ejecución ───
TOMA_GUARDADA = guardar_toma_elegida(toma_elegida, ULTIMA_GENERACION, descargar_archivo=descargar_tambien)

### Si algo no sale bien

| Problema | Qué hacer |
|---|---|
| «Este entorno no tiene GPU» | **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** y empieza otra vez desde la celda 1. |
| «mount failed» o Drive no se conecta | Ejecuta la celda otra vez y, en la ventana de Google, elige **la misma cuenta** con la que abriste Colab y marca **todos los permisos**. Si sigue: *Entorno de ejecución → Desconectar y eliminar entorno de ejecución* y empieza desde la celda 1, o usa el botón de Drive del panel *Archivos*. Mientras tanto, en la celda 4 elige «Subir desde mi computadora». |
| No encuentra tu audio | Escribe la ruta dentro de *Mi unidad*, p. ej. `Audios/mi voz.m4a`. |
| La voz no se parece | Usa 30–60 s de audio limpio, o una carpeta con varios audios tuyos. Activa `reducir_ruido` si se oye ruido de fondo. |
| Sonidos raros, repeticiones o balbuceos | Pide 3 o 4 `tomas` y quédate con la mejor (celda 6). Si se repite, baja `temperatura` (0.5–0.65) o sube `penalizacion_repeticion`. |
| Lee mal una sigla, una abreviatura o un nombre | Agrega una regla en `pronunciacion` (celda 5): `DNI = de ene i`, `Jr. = jirón` (el modelo lee «Jr.» como «junior»), `Av. = avenida`. |
| Lee mal un precio o una cifra | Deja marcada `leer_montos_y_cifras`. Si aun así falla, escribe la cifra sin separadores (`1500`) o en letras. |
| Una palabra suelta en otro idioma suena rara | El modelo lee mejor frases completas: mete unas palabras más dentro de `[en]…[/en]` o pide varias tomas. |
| Necesitas más volumen para redes sociales | En `volumen` (celda 5) elige «Alto: -14 LUFS (redes)». Los picos nunca pasan de −1 dBFS. |
| Te gustó una toma y quieres repetirla | Copia su semilla en el campo `semilla`, deja `tomas` en 1 y vuelve a generar. |
| Textos muy largos (más de unos 20 minutos de audio) | Divídelos en varios `.txt` (por ejemplo, uno por capítulo): si Colab se desconecta a mitad de camino, no pierdes todo. |
| La instalación falla tras una actualización de Colab | En la celda 1 agrega a `PAQUETES_EXTRA` la versión que indique el error (p. ej. `"transformers<5.19"`), usa **Entorno de ejecución → Reiniciar sesión** y vuelve a empezar. |

Los audios generados quedan en `/content/clonador_voz/salidas` (se borran al cerrar la sesión) y, si lo activaste, también en la carpeta de Drive que elegiste.